### Объект Series  

Объект Series во многом напоминает словарь

In [ ]:
import pandas as pd
import numpy as np

s = pd.Series(np.arange(5), index=["a", "b", "c", "d", "e"])
print(s)

s = pd.Series(np.linspace(0, 1, 5))
print(s)
# Вывод программы
# a    0
# b    1
# c    2
# d    3
# e    4
# dtype: int32

# 0    0.00
# 1    0.25
# 2    0.50
# 3    0.75
# 4    1.00
# dtype: float64

Если data — словарь, а index не задан, то в качестве меток используются ключи словаря:

In [ ]:
d = {"a": 10, "b": 20, "c": 30, "g": 40}
print(pd.Series(d))

# Вывод программы:
# a    10
# b    20
# c    30
# g    40
# dtype: int64

Если передать index, не совпадающий с ключами словаря, то отсутствующие значения будут заменены на NaN:

In [ ]:
d = {"a": 10, "b": 20, "c": 30, "g": 40}
print(pd.Series(d, index=["a", "b", "c", "d"]))

# Вывод программы
# a    10.0
# b    20.0
# c    30.0
# d     NaN
# dtype: float64

Если data — число, обязательно передаётся index, определяющий количество элементов:

In [ ]:
index = ["a", "b", "c"]

print(pd.Series(5, index=index))

# Вывод программы
# a    5
# b    5
# c    5
# dtype: int64

Объекты Series поддерживают индексирование, срезы и математические операции:

In [ ]:
s = pd.Series(np.arange(5), index=["a", "b", "c", "d", "e"])
print(s["a"])             # Один элемент
print(s[["a", "d"]])      # Несколько элементов
print(s[1:])              # Срез
print(s + s)              # Поэлементное сложение
print(s[s > 2])           # Фильтровать по условию

Атрибуты Series (name — название набора данных, index.name — имя оси индексов)

In [ ]:
s.name = "Данные"
s.index.name = "Индекс"
print(s)

# Вывод программы
# Индекс
# a    0
# b    1
# c    2
# d    3
# e    4
# Name: Данные, dtype: int32

### Объект DataFrame  

Создать DataFrame можно из словаря списков

In [ ]:
students_marks_dict = {
    "student": ["Студент_1", "Студент_2", "Студент_3"],
    "math": [5, 3, 4],
    "physics": [4, 5, 5]
}
df_students = pd.DataFrame(students_marks_dict)
print(df_students)

# Вывод программы
# student  math  physics
# 0  Студент_1     5        4
# 1  Студент_2     3        5
# 2  Студент_3     4        5

Параметр index задаёт метки осей, передаётся в виде списка. (str[], int[])

In [ ]:
print(df_students.index)   # RangeIndex(start=0, stop=3, step=1)
print(df_students.columns) # Index(['student', 'math', 'physics'], dtype='object')

df_students.index = ["A", "B", "C"]
print(df_students.index)   # Index(['A', 'B', 'C'], dtype='object')

Каждый столбец DataFrame -> отдельный объект Series

In [ ]:
print(type(df_students["student"]))  

# <class 'pandas.core.series.Series'>

Для доступа к строкам по индексам используют .loc[], включая срезы:

In [ ]:
print(df_students.loc["B":])

# Вывод программы
# student  math  physics
# B  Студент_2     3        5
# C  Студент_3     4        5

Если data — это одно значение, параметр index обязательно нужен

In [ ]:
index = ["a", "b", "c"]
print(pd.Series(5, index=index))

# Вывод
# a    5
# b    5
# c    5
# dtype: int64

### Файлы, табличные структуры

#### CSV-файлы *<- самый популярный*

- Считывание: pd.read_csv("имя_файла.csv")  

- Сохранение: df.to_csv("имя_файла.csv")

#### Excel-файлы (формат 2007+)

- Считывание: pd.read_excel("имя_файла.xlsx")  

- Сохранение: df.to_excel("имя_файла.xlsx")  

#### JSON-файлы

- Считывание: pd.read_json("имя_файла.json")  

- Сохранение: df.to_json("имя_файла.json")

### Получение данных  
head(), tail(), `срезы[:]`, фильтрацией `df[df[key] == value]`, сортировкой `sort_values()`

In [ ]:
students = pd.read_csv("test.csv")
print(type(students))  # <class 'pandas.core.frame.DataFrame'>
print(students.head(2))
print(students.tail(1))
print(students[0:2])
print(students[students["test preparation course"] == "completed"]["math score"].head())
print(students[students["test preparation course"] == "completed"]["math score"].sort_values(ascending=False))

# Более эффективный. В памяти не создаются лишние промежуточные таблицы
filtered_scores = (
    students[
        (students["test preparation course"] == "completed") &
        (students["math score"] < 80)
    ]
    ["math score"]
    .sort_values(ascending=False)
)
print(filtered_scores.head())

# Менее эффективный. Если в исходной таблице 50 колонок, они все будут скопированы и отсортированы.
completed_students = students[students["test preparation course"] == "completed"]
print(completed_students.sort_values(by="math score", ascending=False)["math score"])

### Добавить колонку `assign()`  

Метод создает копию исходной таблицы и  добавляет в неё новый столбец `total_score` (общий балл)

Программа идет по каждой строчке, берет оценку за математику, прибавляет к ней чтение, прибавляет письмо и записывает сумму в total_score:

In [ ]:
# SENIOR подход, копия students
scores = students.assign(total_score=lambda x: x["math score"]
                                          + x["reading score"]
                                          + x["writing score"])
print(scores.sort_values("total_score", ascending=False).head())



# JUNIOR подход, портит исходную таблицу students
students["total_score"] = students["math score"] + students["reading score"] + students["writing score"]
print(students.sort_values("total_score", ascending=False).head())

- Без `lambda` приходится внутри скобок каждый раз явно указывать имя исходной таблицы: students["math score"] + students["reading score"]...

- Вариант с `lambda` незаменим в сложных цепочках, когда вы сначала отфильтровали таблицу, а потом _сразу же_ хотите посчитать сумму. В таком случае `lambda` будет брать данные из уже отфильтрованной на лету таблицы, а прямое указание `students[...]` брало бы данные из старой, неотфильтрованной версии.

### Метод `groupby()` позволяет сгруппировать записи по признаку

In [ ]:
# Таблица разделяется на мини-группы по комбинации двух признаков (Пол + Курс)
# На выходе получаются 4 группы
# 1. female + none
# 2. female + completed
# 3. male + none
# 4. male + completed
print(students.groupby(["gender", "test preparation course"])["writing score"].count())

Для вычисления сводных значений используем агрегирующие функции (среднее, медиана, сумма и т. п.).

In [ ]:
# Ключ ("math score") — это название столбца, с которым мы будем работать.
# Значение (["mean", "median"]) — это список функций
agg_functions = {"math score": ["mean", "median"]}

print(students.groupby(["gender", "test preparation course"]).agg(agg_functions))

### Построение графиков

In [ ]:
import matplotlib.pyplot as plt
students = pd.read_csv("test.csv")
# 1. Считаем среднее значение заранее
mean_score = students["math score"].mean()

# 2. Строим гистограмму распределения
plt.hist(students["math score"], bins=20, alpha=0.7, label="Все оценки", color="teal")

# 3. Добавляем вертикальную линию среднего балла (axvline)
plt.axvline(mean_score, color="red", linestyle="dashed", linewidth=2, label=f"Среднее: {mean_score:.1f}")

# Настраиваем график
plt.legend()
plt.title("Оценки по математике со средней линией")
plt.xlabel("Баллы")
plt.ylabel("Количество студентов")
plt.show()


### Задача длины всех слов по четности  

1. Middle-решение (Чистый Python + базовый Pandas)

In [ ]:
import pandas as pd

# Когда использовать: до нескольких тысяч символов текста
def length_stats_middle(text: str) -> tuple[pd.Series, pd.Series]:
    # Посимвольная очистка и перевод в нижний регистр
    clean_text = "".join(
        c.lower() for c in text if c.isalpha() or c.isspace()
    )

    # Удаление дубликатов и сортировка
    words = sorted(list(set(clean_text.split())))

    # Создание серии и фильтрация маской Pandas
    data = pd.Series(map(len, words), index=words, dtype="int64")

    is_odd = data % 2 != 0
    return data[is_odd], data[~is_odd]

2. Senior Pythonic-решение (Регулярные выражения re + Pandas)  
- `Абсолютный лидер по скорости`, очистка текста делегируется встроенному движку регулярных выражений, который работает на уровне скомпилированного Си-кода.

In [ ]:
import re
import pandas as pd

# Когда использовать: Если данные пришли в функцию как текстовая строка (str)
def length_stats_senior_re(text: str) -> tuple[pd.Series, pd.Series]:
    # Моментальный поиск всех слов из букв на уровне Си
    words = re.findall(r"[a-zа-яё]+", text.lower())

    # Быстрое удаление дубликатов и сортировка напрямую из множества
    unique_words = sorted(set(words))

    # Сборка итоговой серии и разделение инверсией маски (~)
    data = pd.Series(map(len, unique_words), index=unique_words, dtype="int64")

    is_odd = data % 2 != 0
    return data[is_odd], data[~is_odd]


3. Senior Векторизованное решение (Полный конвейер .str в Pandas)  
- Вся логика завязана на встроенные строковые методы Pandas

In [ ]:
import pandas as pd

# Когда использовать: Если данные пришли как столбец таблицы (pd.Series / pd.DataFrame)
def length_stats_senior_pandas(text_column: pd.Series) -> tuple[pd.Series, pd.Series]:
    # Столбец таблицы обрабатывается конвейером без циклов
    words = text_column.str.lower().str.findall(r"[a-zа-яё]+").explode()

    # Векторизованное удаление дубликатов и сортировка
    unique_words = words.drop_duplicates().sort_values()

    # Вычисление длин слов на уровне Си-кода Pandas
    data = unique_words.str.len().astype("int64")
    data.index = unique_words

    # Разделение по четности
    is_odd = data % 2 != 0
    return data[is_odd], data[~is_odd]
